# 面试问题：NER 与 Entity Linking 怎样从 span 识别走到唯一知识库实体？

可以直接复述的回答是：NER 先输出文本中的实体 span、类型与字符偏移，Entity Linking 再为 mention 生成候选并结合上下文、先验和类型选择 KB ID。字符串最长匹配是可解释 baseline，但必须解决重叠 span；“苹果”和“亚马逊”也不能仅凭流行度链接。候选排序应输出每个实体的先验、上下文重合和最终分数，低置信度时允许 NIL。训练和评估需区分 span F1 与 linking accuracy。下面用八句新闻/生活文本手写最长非重叠 NER、候选生成和上下文消歧。

## 真实案例：公司、地点、学校与普通名词的歧义链接

八句文本覆盖苹果公司/水果、亚马逊公司/河流，以及“上海”与“上海交通大学”的重叠 mention。知识库和别名为教学构造的小快照。

In [1]:
sentences = [  # 定义八句带人工实体答案的文本
    {"id": "EL-01", "text": "苹果发布新款手机", "gold": [("苹果", "ORG_APPLE")]},  # 苹果公司语境
    {"id": "EL-02", "text": "早餐吃了一个苹果", "gold": [("苹果", "FOOD_APPLE")]},  # 水果语境
    {"id": "EL-03", "text": "亚马逊推出新的云服务", "gold": [("亚马逊", "ORG_AMAZON")]},  # 科技公司语境
    {"id": "EL-04", "text": "亚马逊河流域出现洪水", "gold": [("亚马逊河", "RIVER_AMAZON")]},  # 河流地点语境
    {"id": "EL-05", "text": "上海交通大学公布招生计划", "gold": [("上海交通大学", "UNI_SJTU")]},  # 学校全称重叠语境
    {"id": "EL-06", "text": "上海今天发布暴雨预警", "gold": [("上海", "CITY_SHANGHAI")]},  # 城市实体
    {"id": "EL-07", "text": "微软在北京设立实验室", "gold": [("微软", "ORG_MICROSOFT"), ("北京", "CITY_BEIJING")]},  # 公司和城市双实体
    {"id": "EL-08", "text": "我用苹果手机访问亚马逊购物", "gold": [("苹果", "ORG_APPLE"), ("亚马逊", "ORG_AMAZON")]},  # 两个公司实体
]  # 结束八句文本
knowledge_base = {  # 定义候选实体元数据与上下文关键词
    "ORG_APPLE": {"name": "苹果公司", "type": "ORG", "aliases": ["苹果"], "popularity": 0.90, "keywords": {"发布", "手机", "科技", "设备"}},  # 苹果公司候选
    "FOOD_APPLE": {"name": "苹果水果", "type": "FOOD", "aliases": ["苹果"], "popularity": 0.45, "keywords": {"早餐", "吃", "水果", "一个"}},  # 苹果水果候选
    "ORG_AMAZON": {"name": "亚马逊公司", "type": "ORG", "aliases": ["亚马逊"], "popularity": 0.85, "keywords": {"云", "服务", "购物", "推出"}},  # 亚马逊公司候选
    "RIVER_AMAZON": {"name": "亚马逊河", "type": "LOC", "aliases": ["亚马逊", "亚马逊河"], "popularity": 0.40, "keywords": {"河", "流域", "洪水", "雨林"}},  # 亚马逊河候选
    "UNI_SJTU": {"name": "上海交通大学", "type": "ORG", "aliases": ["上海交通大学", "交大"], "popularity": 0.70, "keywords": {"招生", "大学", "计划", "校园"}},  # 上海交通大学候选
    "CITY_SHANGHAI": {"name": "上海", "type": "LOC", "aliases": ["上海"], "popularity": 0.95, "keywords": {"今天", "暴雨", "城市", "预警"}},  # 上海城市候选
    "ORG_MICROSOFT": {"name": "微软", "type": "ORG", "aliases": ["微软"], "popularity": 0.88, "keywords": {"实验室", "软件", "设立"}},  # 微软公司候选
    "CITY_BEIJING": {"name": "北京", "type": "LOC", "aliases": ["北京"], "popularity": 0.92, "keywords": {"实验室", "城市", "设立"}},  # 北京城市候选
}  # 结束教学知识库
alias_to_entities = {}  # 建立 mention 别名到候选实体列表映射
for entity_id, entity in knowledge_base.items():  # 遍历八个知识库实体
    for alias in entity["aliases"]:  # 遍历实体全部别名
        alias_to_entities.setdefault(alias, []).append(entity_id)  # 将实体加入别名候选桶
print("输入预览：id | text | gold")  # 输出八句标注表头
for sentence in sentences:  # 逐句展示实体链接任务
    print(f"{sentence['id']} | {sentence['text']} | {sentence['gold']}")  # 展示可读文本和人工 KB ID
print("歧义别名候选：苹果=", alias_to_entities["苹果"], "亚马逊=", alias_to_entities["亚马逊"])  # 展示链接而非纯 NER 的必要性

输入预览：id | text | gold
EL-01 | 苹果发布新款手机 | [('苹果', 'ORG_APPLE')]
EL-02 | 早餐吃了一个苹果 | [('苹果', 'FOOD_APPLE')]
EL-03 | 亚马逊推出新的云服务 | [('亚马逊', 'ORG_AMAZON')]
EL-04 | 亚马逊河流域出现洪水 | [('亚马逊河', 'RIVER_AMAZON')]
EL-05 | 上海交通大学公布招生计划 | [('上海交通大学', 'UNI_SJTU')]
EL-06 | 上海今天发布暴雨预警 | [('上海', 'CITY_SHANGHAI')]
EL-07 | 微软在北京设立实验室 | [('微软', 'ORG_MICROSOFT'), ('北京', 'CITY_BEIJING')]
EL-08 | 我用苹果手机访问亚马逊购物 | [('苹果', 'ORG_APPLE'), ('亚马逊', 'ORG_AMAZON')]
歧义别名候选：苹果= ['ORG_APPLE', 'FOOD_APPLE'] 亚马逊= ['ORG_AMAZON', 'RIVER_AMAZON']


## Baseline / 基线：所有别名子串都输出，再选最高流行度实体

基线会在“上海交通大学”中同时输出“上海”和学校全称；对“早餐吃苹果”则因公司更流行而错误链接。

In [2]:
def all_substring_mentions(text):  # 实现不处理重叠的别名子串匹配
    spans = []  # 收集所有命中的 mention span
    for alias in alias_to_entities:  # 遍历知识库全部别名
        start = text.find(alias)  # 查找当前别名首次出现位置
        if start >= 0:  # 检查别名是否出现在句子中
            spans.append((start, start + len(alias), alias))  # 保存字符半开区间和 mention
    return sorted(spans)  # 按字符位置返回所有重叠结果
def popularity_link(mention):  # 只按实体先验选择候选
    candidates = alias_to_entities[mention]  # 获取 mention 对应 KB 候选
    return max(candidates, key=lambda entity_id: knowledge_base[entity_id]["popularity"])  # 选择流行度最高实体
baseline_links = []  # 收集八句流行度链接结果
print("id | baseline spans | linked IDs")  # 输出基线结果表头
for sentence in sentences:  # 遍历八句输入文本
    spans = all_substring_mentions(sentence["text"])  # 获取可能重叠的全部字符串 span
    links = [(mention, popularity_link(mention)) for start, end, mention in spans]  # 对每个 mention 选最流行实体
    baseline_links.append(links)  # 保存当前句基线链接
    print(f"{sentence['id']} | {spans} | {links}")  # 展示重叠和歧义错误

id | baseline spans | linked IDs
EL-01 | [(0, 2, '苹果')] | [('苹果', 'ORG_APPLE')]
EL-02 | [(6, 8, '苹果')] | [('苹果', 'ORG_APPLE')]
EL-03 | [(0, 3, '亚马逊')] | [('亚马逊', 'ORG_AMAZON')]
EL-04 | [(0, 3, '亚马逊'), (0, 4, '亚马逊河')] | [('亚马逊', 'ORG_AMAZON'), ('亚马逊河', 'RIVER_AMAZON')]
EL-05 | [(0, 2, '上海'), (0, 6, '上海交通大学')] | [('上海', 'CITY_SHANGHAI'), ('上海交通大学', 'UNI_SJTU')]
EL-06 | [(0, 2, '上海')] | [('上海', 'CITY_SHANGHAI')]
EL-07 | [(0, 2, '微软'), (3, 5, '北京')] | [('微软', 'ORG_MICROSOFT'), ('北京', 'CITY_BEIJING')]
EL-08 | [(2, 4, '苹果'), (8, 11, '亚马逊')] | [('苹果', 'ORG_APPLE'), ('亚马逊', 'ORG_AMAZON')]


## 核心实现：最长非重叠 span 与上下文候选评分

先按 alias 长度降序提出 span，再拒绝字符区间重叠。链接分数为 `0.4*popularity + 0.6*context_overlap`，并输出所有候选分项。

In [3]:
def longest_non_overlapping_mentions(text):  # 手写最长优先的非重叠 span 识别
    proposals = []  # 收集所有别名在文本中的位置
    for alias in alias_to_entities:  # 遍历全部知识库别名
        search_start = 0  # 初始化当前别名查找起点
        while True:  # 支持同一别名在文本中多次出现
            start = text.find(alias, search_start)  # 从当前位置寻找下一个匹配
            if start < 0:  # 检查是否已无更多匹配
                break  # 结束当前别名查找
            proposals.append((start, start + len(alias), alias))  # 保存候选字符 span
            search_start = start + 1  # 向后移动避免重复同一位置
    proposals.sort(key=lambda span: (-(span[1] - span[0]), span[0], span[2]))  # 按长度降序再按位置排序
    selected = []  # 收集不重叠最终 span
    occupied = set()  # 记录已经被长 mention 占用的字符位置
    for start, end, mention in proposals:  # 按最长优先遍历候选
        positions = set(range(start, end))  # 构造当前 span 字符位置集合
        if occupied & positions:  # 检查是否与已选长 span 重叠
            continue  # 跳过短的重叠 mention
        selected.append((start, end, mention))  # 接受当前实体 span
        occupied |= positions  # 标记字符区间已经占用
    return sorted(selected)  # 按文本阅读顺序返回 NER 结果
def link_candidates(text, mention):  # 为一个 mention 计算上下文消歧分数
    scored = []  # 收集候选及各分项
    context_characters = set(text)  # 使用字符集合形成轻量上下文特征
    for entity_id in alias_to_entities[mention]:  # 遍历 mention 的全部 KB 候选
        entity = knowledge_base[entity_id]  # 读取候选元数据
        keyword_hits = [keyword for keyword in entity["keywords"] if keyword in text]  # 查找候选上下文关键词
        context_score = len(keyword_hits) / max(1, len(entity["keywords"]))  # 按关键词集合大小归一化
        total_score = 0.4 * entity["popularity"] + 0.6 * context_score  # 合并先验和上下文证据
        scored.append({"entity_id": entity_id, "popularity": entity["popularity"], "hits": keyword_hits, "context": context_score, "score": total_score})  # 保存可审计候选分数
    scored.sort(key=lambda item: (-item["score"], item["entity_id"]))  # 按总分降序稳定排序
    return scored  # 返回全部候选而非只给最终 ID
example_spans = longest_non_overlapping_mentions(sentences[4]["text"])  # 处理上海交通大学重叠反例
fruit_candidates = link_candidates(sentences[1]["text"], "苹果")  # 处理苹果水果消歧反例
print("上海交通大学最长非重叠 spans：", example_spans)  # 展示短城市 mention 被长学校覆盖
print("早餐苹果候选：entity | popularity | keyword_hits | context | total")  # 输出实体链接分项表头
for candidate in fruit_candidates:  # 遍历苹果两个候选实体
    print(candidate)  # 展示上下文如何克服公司流行度先验

上海交通大学最长非重叠 spans： [(0, 6, '上海交通大学')]
早餐苹果候选：entity | popularity | keyword_hits | context | total
{'entity_id': 'FOOD_APPLE', 'popularity': 0.45, 'hits': ['早餐', '吃', '一个'], 'context': 0.75, 'score': 0.63}
{'entity_id': 'ORG_APPLE', 'popularity': 0.9, 'hits': [], 'context': 0.0, 'score': 0.36000000000000004}


## 八句逐 span 结果与指标

In [4]:
predicted_links = []  # 收集八句最终 mention 与实体链接
gold_total = 0  # 统计人工实体总数
correct_total = 0  # 统计 mention 与 KB ID 同时正确数量
print("id | span(start,end,text) | linked_entity | gold | correct")  # 输出逐实体结果表头
for sentence in sentences:  # 遍历八句文本
    spans = longest_non_overlapping_mentions(sentence["text"])  # 提取最长非重叠 mention
    sentence_links = []  # 收集当前句预测实体
    for start, end, mention in spans:  # 遍历当前句识别 span
        candidates = link_candidates(sentence["text"], mention)  # 生成并排序 KB 候选
        linked = candidates[0]["entity_id"]  # 选择最高分实体
        sentence_links.append((mention, linked))  # 保存 mention 与 KB ID
        gold_match = (mention, linked) in sentence["gold"]  # 检查 span 文本和链接是否同时正确
        correct_total += int(gold_match)  # 累加端到端正确实体
        print(f"{sentence['id']} | ({start},{end},{mention}) | {linked} | {sentence['gold']} | {gold_match}")  # 展示字符偏移和消歧结果
    gold_total += len(sentence["gold"])  # 累加人工实体数量
    predicted_links.append(sentence_links)  # 保存句级预测
linking_accuracy = correct_total / gold_total  # 计算端到端实体正确率
baseline_correct = sum(int(link in sentence["gold"]) for links, sentence in zip(baseline_links, sentences) for link in links)  # 统计流行度基线正确链接
baseline_precision = baseline_correct / sum(len(links) for links in baseline_links)  # 计算基线含重叠 span 的实体精度
print(f"baseline entity precision={baseline_precision:.1%}，longest+context accuracy={linking_accuracy:.1%}")  # 对比同八句结果

id | span(start,end,text) | linked_entity | gold | correct
EL-01 | (0,2,苹果) | ORG_APPLE | [('苹果', 'ORG_APPLE')] | True
EL-02 | (6,8,苹果) | FOOD_APPLE | [('苹果', 'FOOD_APPLE')] | True
EL-03 | (0,3,亚马逊) | ORG_AMAZON | [('亚马逊', 'ORG_AMAZON')] | True
EL-04 | (0,4,亚马逊河) | RIVER_AMAZON | [('亚马逊河', 'RIVER_AMAZON')] | True
EL-05 | (0,6,上海交通大学) | UNI_SJTU | [('上海交通大学', 'UNI_SJTU')] | True
EL-06 | (0,2,上海) | CITY_SHANGHAI | [('上海', 'CITY_SHANGHAI')] | True
EL-07 | (0,2,微软) | ORG_MICROSOFT | [('微软', 'ORG_MICROSOFT'), ('北京', 'CITY_BEIJING')] | True
EL-07 | (3,5,北京) | CITY_BEIJING | [('微软', 'ORG_MICROSOFT'), ('北京', 'CITY_BEIJING')] | True
EL-08 | (2,4,苹果) | ORG_APPLE | [('苹果', 'ORG_APPLE'), ('亚马逊', 'ORG_AMAZON')] | True
EL-08 | (8,11,亚马逊) | ORG_AMAZON | [('苹果', 'ORG_APPLE'), ('亚马逊', 'ORG_AMAZON')] | True
baseline entity precision=75.0%，longest+context accuracy=100.0%


## 失败案例与修正：流行度偏置和重叠 span

“早餐吃苹果”若只看先验会链接到公司；“上海交通大学”若输出所有子串会额外产生城市实体。上下文关键词修正前者，最长非重叠选择修正后者。

In [5]:
fruit_popularity_entity = popularity_link("苹果")  # 获取不看上下文的苹果最高先验实体
fruit_context_entity = fruit_candidates[0]["entity_id"]  # 获取早餐语境下最高分实体
overlapping_baseline = all_substring_mentions("上海交通大学公布招生计划")  # 复现学校内部城市重叠
non_overlapping_fixed = longest_non_overlapping_mentions("上海交通大学公布招生计划")  # 执行最长 span 修正
print("苹果仅先验：", fruit_popularity_entity, "加入早餐上下文：", fruit_context_entity)  # 展示实体消歧失败与修正
print("重叠 baseline：", overlapping_baseline)  # 展示同时输出上海和学校
print("最长非重叠：", non_overlapping_fixed)  # 展示只保留学校全称
print("BIO 示例：", [(character, "B-ORG" if index == 0 else "I-ORG") for index, character in enumerate("上海交通大学")])  # 展示长实体字符标签边界

苹果仅先验： ORG_APPLE 加入早餐上下文： FOOD_APPLE
重叠 baseline： [(0, 2, '上海'), (0, 6, '上海交通大学')]
最长非重叠： [(0, 6, '上海交通大学')]
BIO 示例： [('上', 'B-ORG'), ('海', 'I-ORG'), ('交', 'I-ORG'), ('通', 'I-ORG'), ('大', 'I-ORG'), ('学', 'I-ORG')]


## 结果解读

NER 与 Linking 是两个错误来源：span 错了就没有正确候选，候选对但消歧错也会给出错误 KB ID。最长匹配解决了教学重叠，上下文分项则让低流行度水果和河流实体胜出。所有结果都保留字符 offset 和候选证据。

## 生产边界

词典规则无法覆盖新实体、别名变体和嵌套实体。生产系统通常用序列标注/生成模型做 NER，用向量召回加 cross-encoder 做链接，并支持 NIL 与知识库版本。评估应分别报告 span exact/partial F1、candidate recall 和 linking accuracy，且按实体类型、流行度和新鲜度切片。

## 最小回归测试

In [6]:
assert len(sentences) >= 6 and len(knowledge_base) >= 6  # 保证案例包含多个文本和实体
assert len(overlapping_baseline) == 2 and len(non_overlapping_fixed) == 1  # 保证重叠 span 失败与修正真实发生
assert non_overlapping_fixed[0][2] == "上海交通大学"  # 保证最长实体获得正确字符边界
assert fruit_popularity_entity == "ORG_APPLE"  # 保证流行度基线误链公司
assert fruit_context_entity == "FOOD_APPLE"  # 保证早餐上下文修正为水果实体
assert linking_accuracy == 1.0  # 保证八句端到端教学实体全部正确
assert linking_accuracy > baseline_precision  # 保证最长 span 与上下文方案优于流行度基线